In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))



In [3]:
from src.data.validation import build_manifest

DATASET_DIR = Path("../data/raw/classification").resolve()

manifest = build_manifest(DATASET_DIR)

OUTPUT_PATH = Path("../artifacts/classification/manifest.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

manifest.to_csv(OUTPUT_PATH, index=False)


In [4]:
from src.data.splitting import split_dataset , export_classification_splits

train_df, val_df, test_df = split_dataset(manifest)


In [5]:

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{name}: {len(df)} images")
    print(df["label"].value_counts().sort_index())


Train: 4916 images
label
glioma        1135
meningioma    1151
notumor       1400
pituitary     1230
Name: count, dtype: int64

Validation: 1053 images
label
glioma        243
meningioma    247
notumor       300
pituitary     263
Name: count, dtype: int64

Test: 1054 images
label
glioma        243
meningioma    247
notumor       300
pituitary     264
Name: count, dtype: int64


In [6]:
split = {
    "train" : train_df,
    "valid":val_df,
    "test":test_df
}

export_classification_splits(splits=split,output_dir="../data/processed/classification",
)

FileExistsError: Destination déjà existante : ..\data\processed\classification\train\glioma\Te-gl_0284.jpg

In [7]:
from src.classification.dataset import load_classification_image

train_ds, val_ds, test_ds, class_names = (
    load_classification_image(data_dir="../data/processed/classification/")
)

print("Classes :", class_names)

for images, labels in train_ds.take(1):
    print("Dimensions des images :", images.shape)
    print("Dimensions des labels :", labels.shape)
    print("Type des images :", images.dtype)

Found 4916 files belonging to 4 classes.
Found 1053 files belonging to 4 classes.
Found 1054 files belonging to 4 classes.
Classes : ['glioma', 'meningioma', 'notumor', 'pituitary']
Dimensions des images : (32, 224, 224, 3)
Dimensions des labels : (32,)
Type des images : <dtype: 'float32'>


In [10]:
from src.classification.pepline  import (
    prepare_train,
    prepare_evaluation,
)

import tensorflow as tf

In [11]:
train_ds = train_ds.map(
    prepare_train,
    num_parallel_calls=tf.data.AUTOTUNE,
)

val_ds = val_ds.map(
    prepare_evaluation,
    num_parallel_calls=tf.data.AUTOTUNE,
)

test_ds = test_ds.map(
    prepare_evaluation,
    num_parallel_calls=tf.data.AUTOTUNE,
)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [12]:
from src.classification.model import build_googlenet

NUM_CLASSES = 4  # À adapter à tes classes réelles

model = build_googlenet(num_classes=NUM_CLASSES)

model.summary()

c:\Users\errak\Desktop\BrainTumorDetectionAI\.venv\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


c:\Users\errak\Desktop\BrainTumorDetectionAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: "brainscan_googlenet"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 image (InputLayer)          [(None, 224, 224, 3)]     0         
                                                                 
 googlenet_feature_extracto  (None, 1024)              5607184   
 r (KerasLayer)                                                  
                                                                 
 classifier_dense (Dense)    (None, 256)               262400    
                                                                 
 classifier_dropout (Dropou  (None, 256)               0         
 t)                                                              
                                                                 
 predictions (Dense)         (None, 4)                 1028      
                                                                 
Total params: 5870612 (22.39 MB)
Trainable para

In [14]:
from src.classification.train import train_googlenet

model, history = train_googlenet(
    model=model,
    train_ds=train_ds,
    valid_ds=val_ds,
    epochs=30,
    output_dir="artifacts/models",
    patience=5,
)

Epoch 1/30
106/154 [===================>..........] - ETA: 44s - loss: 0.6302 - accuracy: 0.7506

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Époque")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Époque")
plt.legend()

plt.tight_layout()
plt.show()

Python : 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
tensorflow: 2.22.0rc0
tensorflow-hub: 0.16.1
tf-keras: 2.15.0
kagglehub: 1.0.2
